## Modèle de base - Random Forest après update preprocessing

In [ ]:
import os
import sys
parent_dir = os.path.abspath('..')
sys.path.append(parent_dir) #set root to path so src visible

# Import des librairies

## Data loading
import s3fs
import time
import pandas as pd
import logging
logging.basicConfig(level=logging.INFO)  #to be able to see the logging info+warnings(to only see warnings: logging.DEBUG)

## Traitement des emails
from src.model.topicModeling import EmailBERTopicNoLeakagePipeline

## Data preprocessing
from src.data_processing.data_load import data_loading
from src.data_processing.data_processing import DataProcessor
from src.utils.TransformData import dataframe_merge, code_insee_from_communes_as_dict

## ML + evaluation
!python -m spacy download fr_core_news_md
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score, classification_report, confusion_matrix, roc_curve
from sklearn.model_selection import GridSearchCV, StratifiedKFold

In [4]:
#1. Pre-process training data (scaler and encoder are saved during this process)
train_processor = DataProcessor(mode="training")
df_train = train_processor.run(optional_encoding=True, optional_normalisation=False)  #fit training set and save arguments (scaler, ordinal map, target_encoding_maps, ...)

#2. Pre-process test data (by using the same scaler/encoder fitted on training dataset)
test_processor = DataProcessor(mode="validation")

#  HERE:  Reuse the same pre-processor fitted on training dataset (so no data-leakage with target-encoding, normalisation...)
test_processor.global_means = train_processor.global_means
test_processor.target_encoding_maps = train_processor.target_encoding_maps
test_processor.fixed_categories_fitted = train_processor.fixed_categories_fitted
test_processor.ordinal_maps          = train_processor.ordinal_maps
test_processor.encoded_columns       = train_processor.encoded_columns
test_processor.scaler                = train_processor.scaler
test_processor.normalized_columns    = train_processor.normalized_columns #only if optional_normalisation = True

df_test = test_processor.run_transform(optional_encoding=True, optional_normalisation=False) #pre-process test data using same args 

INFO:src.data_processing.data_processing:Dropped columns from dataset: ['client_code_postal', 'client_structure_familiale', 'client_nom_banque', 'client_nps_verbatim_n', 'client_nps_verbatim_n_moins1', 'impaye_nb_actions', 'impaye_montant_total', 'impaye_duree_max_action_jours', 'impaye_action_1er_impaye', 'impaye_action_1ere_lettre_relance', 'impaye_action_2eme_impaye', 'impaye_action_annonce_contentieux', 'impaye_action_en_recouvrement', 'impaye_action_mise_en_demeure', 'impaye_action_suspension_garanties', 'client_nps_date_reponse_n_moins1_jours', 'client_nps_date_reponse_n_jours']
INFO:src.data_processing.data_processing:Columns that are Fixed-category one-hot encoded: ['courtier_segmentation_interne']
INFO:src.data_processing.data_processing:Columns that are One-hot encoded: ['client_mode_paiement', 'courtier_reseau_courtage', 'courtier_type_commission']
INFO:src.data_processing.data_processing:Columns that are Target encoded: ['client_ro_souscripteur', 'client_produit', 'client_g

In [ ]:
#3. Preprocess et dit du BERTOPIC sur le train
emailPreprocessor = EmailBERTopicNoLeakagePipeline()
_, _, _, email_training_dataset, _ = data_loading("training")
email_training_dataset["objet_contenu"] = email_training_dataset["interaction_texte_mail"] 
email_train = emailPreprocessor.build_dataset_split(
    df_split=email_training_dataset,
    email_mode="fit"
)


#4. Jointure à la base de train


#5. Preprocess et transform du BERTOPIC sur les données de validation
_, _, _, email_validation_dataset, _ = data_loading("validation")
email_validation_dataset["objet_contenu"] = email_validation_dataset["interaction_texte_mail"] 
email_validation = emailPreprocessor.build_dataset_split(
    df_split=email_validation_dataset,
    email_mode="transform"
)


#6. Jointure à la base de validation

INFO:sentence_transformers.base.model:No device provided, using cpu


INFO:httpx:HTTP Request: HEAD https://huggingface.co/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2/e8f8c211226b894fcb81acc59f3b34ba3efd5f42/modules.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2/e8f8c211226b894fcb81acc59f3b34ba3efd5f42/config_sentence_transformers.json "HTTP/1.1 200 OK"
INFO:sentence_transformers.base.model:Loading SentenceTransformer model from sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2.
INFO:httpx:HTTP Request: HEAD https://huggingface.co/s

entame du cleaning complet avec les regex et spacy
Fin du cleaning complet avec les regex et spacy


Batches: 100%|██████████| 1506/1506 [44:39<00:00,  1.78s/it] 
2026-04-14 14:08:34,234 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-04-14 14:13:41,369 - BERTopic - Dimensionality - Completed ✓
2026-04-14 14:13:41,373 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-04-14 14:34:15,535 - BERTopic - Cluster - Completed ✓
2026-04-14 14:34:15,570 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-04-14 14:34:18,274 - BERTopic - Representation - Completed ✓


,client_code,interaction_motif,interaction_precision,interaction_canal,interaction_service,interaction_est_transferee_service_reclamation,interaction_est_transferee_service_gestion,interaction_est_transferee_service_commercial,interaction_est_externalisee,interaction_texte_mail,...,topic_prop_450,topic_prop_451,topic_prop_452,topic_prop_453,topic_prop_454,topic_prop_455,topic_prop_456,topic_prop_457,dominant_topic,latest_topic
0,22037,Décès,"Explications (procédures, garanties)",E-mail,Prestations prévoyance,0.0,0.5,0.0,0.5,xxpj0 xxobj Adhérent 3148 Mme flag_nom_ xxc...,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-1.0,-1.0
1,22331,Optique,"Explication (prestations, devis, indu)",Téléphone,Prestations santé,0.0,0.5,0.0,0.7,None,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-99.0,-99.0
2,22702,Demandes générales,Demande de document - Autre,Téléphone,Prestations santé,0.0,0.5,0.0,1.0,None,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-99.0,-99.0
3,23003,Frais courants,"Explication (prestations, devis, indu)",Téléphone,Prestations santé,0.0,0.0,0.0,1.0,None,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-99.0,-99.0
4,23164,Frais courants,"Explication (prestations, devis, indu)",Téléphone,Prestations santé,0.0,1.0,0.0,1.0,None,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-99.0,-99.0


In [ ]:
#5. Preprocess et transform du BERTOPIC sur les données de validation
_, _, _, email_validation_dataset, _ = data_loading("validation")
email_validation_dataset["objet_contenu"] = email_validation_dataset["interaction_texte_mail"] 
email_validation = emailPreprocessor.build_dataset_split(
    df_split=email_validation_dataset,
    email_mode="transform"
)

In [ ]:
email_validation_dataset["objet_contenu"] = email_validation_dataset["interaction_texte_mail"] 
email_validation = emailPreprocessor.build_dataset_split(
    df_split=email_validation_dataset,
    email_mode="transform"
)

entame du cleaning complet avec les regex et spacy
Fin du cleaning complet avec les regex et spacy
